# Models

In [1]:
from pathlib import Path
import sys

project_root = Path.cwd()
if not (project_root / 'main.py').exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd

from config import config
from preprocessing import Preprocessor, remove_outliers
from utils import get_best_score

from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.linear_model import LinearRegression

In [2]:
train = pd.read_csv(config.paths.train_data)
test = pd.read_csv(config.paths.test_data)

X = train.drop(columns='SalePrice')
y = np.log(train['SalePrice'])

X, y = remove_outliers(X, y)

preprocessor = Preprocessor()

X_train = preprocessor.fit_transform(X, y)
X_test = preprocessor.transform(test)

print('Train:', X_train.shape)
print('Test:', X_test.shape)

Train: (1458, 37)
Test: (1459, 37)


In [3]:
score_calc = 'neg_root_mean_squared_error'

## Baseline - линейная регрессия

In [4]:
pipeline = Pipeline([
    ('preprocessing', Preprocessor()),
    ('model', LinearRegression())
])

parameters = {
    'model__fit_intercept': [True, False]
}

cv = KFold(
    n_splits=config.split.n_splits,
    shuffle=config.split.shuffle,
    random_state=config.split.random_state
)

grid_linear = GridSearchCV(
    pipeline,
    parameters,
    cv=cv,
    scoring=score_calc,
    verbose=1,
    n_jobs=-1
)

grid_linear.fit(X, y)

get_best_score(grid_linear)

Fitting 5 folds for each of 2 candidates, totalling 10 fits
Best score: 0.14135
Best params: {'model__fit_intercept': True}
Best estimator: Pipeline(steps=[('preprocessing', Preprocessor()),
                ('model', LinearRegression())])


np.float64(0.14134729986002137)

RMSE для бейзлайн решения - 0.14135